# Standalone CV-DV LCHS Notebook: 1D Homogeneous Dirichlet Heat Equation

This notebook reconstructs the active heat-equation workflow in the repository without importing any custom `formed_*` modules. Everything needed for the numerical experiment is written inline so that the notebook stands on its own and directly illustrates the working paper.

## Scope

The PDE of interest is the one-dimensional homogeneous Dirichlet heat equation
$$
\partial_t u(x,t)=\alpha\,\partial_{xx}u(x,t),\qquad u(0,t)=u(1,t)=0.
$$
We discretize the spatial operator and then study the finite-dimensional ODE
$$
\dot{\mathbf u}(t)=-A\,\mathbf u(t),
$$
using the same 2-qubit Dirichlet benchmark that is currently active in the repo.

The notebook implements three CV seed-state preparation modes:

1. `injection`: exact simulator baseline via direct Fock-state injection.
2. `snap_d`: genuine gate-level SNAP+Displacement state preparation.
3. `givens`: analytic Law-Eberly / Givens decomposition. Because `bosonic_qiskit` does not expose a photon-number-selective Jaynes-Cummings primitive, the notebook injects the analytically prepared state and reports the corresponding JC pulse count separately.

## What is exact and what is approximate?

1. The classical reference $\mathbf u(T)=e^{-AT}\mathbf u(0)$ is exact for the chosen finite-dimensional discretization.
2. The hybrid CV-DV circuit is approximate because the joint unitary $e^{-iT\hat x\otimes A}$ is split into first-order Pauli-term Trotter factors.
3. The `injection` seed-prep path is an exact simulator baseline.
4. The `snap_d` path is an actual gate-compiled CV state-preparation routine.
5. The `givens` path is exact as an analytic decomposition, but its circuit realization here is still simulator injection plus separately reported Law-Eberly resource counts.

Throughout the notebook we use the same $\hbar=1$ theory convention used by the active coefficient code:
$$
\hat x = \frac{a+a^\dagger}{\sqrt 2},\qquad
\hat p = \frac{-i(a-a^\dagger)}{\sqrt 2},\qquad
[\hat x,\hat p]=i.
$$
This is why the conditional-displacement amplitude becomes $\alpha = -i\lambda/\sqrt 2$ in the Trotter block.


## Mathematical Setup

For the active benchmark we keep the same encoded Dirichlet Laplacian used by the current bosonic implementation. With four interior grid states ($M=4$), the discretized Laplacian is represented on two qubits as
$$
T_2 = 2I\!I - I\!X - \frac12(X\!X + Y\!Y).
$$
The heat generator is therefore
$$
A = \frac{\alpha}{h^2}T_2,
$$
and the exact discrete solution is
$$
\mathbf u(T)=e^{-AT}\mathbf u(0).
$$

The hybrid CV-DV LCHS circuit uses the joint unitary
$$
U(T)=e^{-iT\hat x\otimes A},
$$
with the oscillator position operator $\hat x=(a+a^\dagger)/\sqrt 2$. The CV sandwich from the paper is implemented as
$$
K(T)=\langle \phi_r|U(T)|\psi_{r,r',\beta}\rangle,
$$
where $|\psi_{r,r',\beta}\rangle$ is the prepared CV seed state followed by a squeeze $S(r')$, and $\langle\phi_r|$ is realized by applying the inverse squeeze and post-selecting the oscillator onto Fock $|0\rangle$.

For circuit construction we use the first-order Lie product formula
$$
e^{-iT\hat x\otimes A}
\approx
\left(\prod_j e^{-i\Delta t\,\hat x\otimes c_j P_j}\right)^{n_{\mathrm{Trot}}},
\qquad
\Delta t = T/n_{\mathrm{Trot}},
$$
where the active Pauli list is
$$
A = c_0 I\!I + c_1 I\!X + c_2 X\!X + c_3 Y\!Y.
$$

Each Pauli factor is compiled by:

1. rotating all non-identity qubits into the $Z$ basis,
2. collecting the parity of the active qubits with a CNOT ladder,
3. applying one conditional displacement on the oscillator,
4. uncomputing the ladder and undoing the basis change.

For the active two-qubit heat block, this gives the same per-step logical counts as the current implementation:
- `1 cv_d`
- `3 cv_c_d`
- `4 cx`
- `10 h`
- `4 rz`


In [ ]:
import math
import time
from dataclasses import dataclass
from typing import Dict, List, Sequence, Tuple

import numpy as np
from scipy.integrate import quad
from scipy.linalg import expm
from scipy.optimize import minimize
from scipy.special import eval_hermite, gammaln

from bosonic_qiskit import CVCircuit, QumodeRegister
from bosonic_qiskit import util as cv_util
from qiskit import QuantumRegister

np.set_printoptions(precision=6, suppress=True)


In [ ]:
# -----------------------------------------------------------------------------
# 1) Discrete heat problem and LCHS coefficient construction
# -----------------------------------------------------------------------------

def pauli_1q(label: str) -> np.ndarray:
    if label == "I":
        return np.array([[1.0, 0.0], [0.0, 1.0]], dtype=complex)
    if label == "X":
        return np.array([[0.0, 1.0], [1.0, 0.0]], dtype=complex)
    if label == "Y":
        return np.array([[0.0, -1.0j], [1.0j, 0.0]], dtype=complex)
    if label == "Z":
        return np.array([[1.0, 0.0], [0.0, -1.0]], dtype=complex)
    raise ValueError(f"Unsupported Pauli label: {label}")


def pauli_string_matrix(pauli: str) -> np.ndarray:
    out = np.array([[1.0]], dtype=complex)
    for ch in pauli:
        out = np.kron(out, pauli_1q(ch))
    return out


def pauli_sum_matrix(pauli_strings: Sequence[str], coeffs: Sequence[complex]) -> np.ndarray:
    dim = 2 ** len(pauli_strings[0])
    out = np.zeros((dim, dim), dtype=complex)
    for p, c in zip(pauli_strings, coeffs):
        out += complex(c) * pauli_string_matrix(p)
    return out


def heat_system_dirichlet_2qubit(alpha: float = 1.0, h_grid: float = 1.0):
    """Active 2-qubit Dirichlet heat benchmark used in the repo."""
    s = alpha / (h_grid ** 2)
    pauli_strings = ["II", "IX", "XX", "YY"]
    coeffs = [2.0 * s, -1.0 * s, -0.5 * s, -0.5 * s]
    A = pauli_sum_matrix(pauli_strings, coeffs)
    return pauli_strings, coeffs, A


def kernel_g_beta(k_points: np.ndarray, beta: float) -> np.ndarray:
    c_beta = 2.0 * np.pi * np.exp(-(2.0 ** beta))
    return np.exp(-((1.0 + 1.0j * k_points) ** beta)) / (c_beta * (1.0 - 1.0j * k_points))


def gamma_hbar1(r_target: float, r_prime: float) -> float:
    return 0.5 * (np.exp(-2.0 * r_prime) - np.exp(-2.0 * r_target))


def fock_prefactor(n: int, sigma_prime: float) -> float:
    return float(
        np.exp(-0.5 * (n * np.log(2.0) + gammaln(n + 1.0)))
        / np.sqrt(np.sqrt(np.pi) * sigma_prime)
    )


def lchs_coefficients_explicit_overlap(
    r_target: float,
    r_prime: float,
    beta: float,
    n_coeff: int,
    n_quad: int = 220,
) -> np.ndarray:
    sigma_prime = float(np.exp(r_prime))
    gamma = gamma_hbar1(r_target, r_prime)
    k_tail = np.sqrt(max(-np.log(1e-14) / max(gamma, 1e-15), 1.0))
    k_max = max(8.0 * sigma_prime, 1.15 * k_tail, 12.0)

    coeffs = np.zeros(n_coeff, dtype=complex)
    for n in range(n_coeff):
        pref = fock_prefactor(n, sigma_prime)

        def integrand(k: float) -> complex:
            return (
                pref
                * eval_hermite(n, k / sigma_prime)
                * kernel_g_beta(np.array([k]), beta)[0]
                * np.exp(-gamma * k * k)
            )

        re = quad(
            lambda k: float(np.real(integrand(k))),
            -k_max,
            k_max,
            limit=max(100, n_quad),
            epsabs=1e-10,
            epsrel=1e-9,
        )[0]
        im = quad(
            lambda k: float(np.imag(integrand(k))),
            -k_max,
            k_max,
            limit=max(100, n_quad),
            epsabs=1e-10,
            epsrel=1e-9,
        )[0]
        coeffs[n] = re + 1.0j * im

    coeffs /= np.linalg.norm(coeffs)
    return coeffs


def classical_heat_solution(A: np.ndarray, total_time: float, init_state: np.ndarray) -> np.ndarray:
    return expm(-A * total_time) @ init_state


def fit_global_scale(observed: np.ndarray, target: np.ndarray) -> Tuple[complex, float]:
    eta = np.vdot(target, observed) / np.vdot(target, target)
    rel_err = np.linalg.norm(observed - eta * target) / max(np.linalg.norm(eta * target), 1e-15)
    return complex(eta), float(rel_err)


def state_fidelity(v1: np.ndarray, v2: np.ndarray) -> float:
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 < 1e-15 or n2 < 1e-15:
        return 0.0
    return float(np.abs(np.vdot(v1 / n1, v2 / n2)) ** 2)


## CV Seed-State Preparation

The LCHS kernel produces a finite set of Fock coefficients $C_n$ that define the oscillator seed state
$$
|\mathrm{seed}\rangle = \sum_{n=0}^{N_{\mathrm{coeff}}-1} C_n |n\rangle.
$$

This notebook keeps two state-preparation stories separate.

### 1. SNAP+D

This is the genuine gate-level compilation route. We alternate
$$
D(\alpha_1)\,\mathrm{SNAP}(\theta_1)\;D(\alpha_2)\,\mathrm{SNAP}(\theta_2)\;\cdots
$$
and optimize the parameters to maximize overlap with the target seed state.

### 2. Law-Eberly / Givens

The analytic decomposition into adjacent Fock-level rotations is exact and physically meaningful, because it tells us how many nontrivial selective Jaynes-Cummings pulses would be required on a platform that supports photon-number selectivity. However, the current `bosonic_qiskit` gate set does not provide that selective primitive. For that reason the notebook:

- computes the exact Givens / Law-Eberly angles,
- reports the number of nontrivial JC pulses,
- injects the analytically prepared state into the simulator.

That is the same honesty standard already used by the active bosonic implementation.


In [ ]:
# -----------------------------------------------------------------------------
# 2) Standalone state-preparation helpers
# -----------------------------------------------------------------------------

def _annihilation_op(n_fock: int) -> np.ndarray:
    a = np.zeros((n_fock, n_fock), dtype=complex)
    for n in range(1, n_fock):
        a[n - 1, n] = np.sqrt(n)
    return a


def displacement_matrix(alpha: complex, n_fock: int) -> np.ndarray:
    a = _annihilation_op(n_fock)
    a_dag = a.T.conj()
    return expm(alpha * a_dag - np.conj(alpha) * a)


def simulate_snap_d_circuit(params: np.ndarray, n_fock: int, depth: int, n_snap: int = 0) -> np.ndarray:
    if n_snap <= 0:
        n_snap = n_fock

    params_per_layer = n_snap + 2
    state = np.zeros(n_fock, dtype=complex)
    state[0] = 1.0

    for layer in range(depth):
        offset = layer * params_per_layer
        thetas = params[offset : offset + n_snap]
        re_alpha = params[offset + n_snap]
        im_alpha = params[offset + n_snap + 1]
        alpha = re_alpha + 1j * im_alpha

        state[:n_snap] *= np.exp(1j * thetas)
        state = displacement_matrix(alpha, n_fock) @ state

    return state


def fidelity_cost(params: np.ndarray, target: np.ndarray, n_fock: int, depth: int, n_snap: int = 0) -> float:
    prepared = simulate_snap_d_circuit(params, n_fock, depth, n_snap)
    return 1.0 - np.abs(np.vdot(target, prepared)) ** 2


@dataclass
class SNAPDResult:
    fidelity: float
    depth: int
    n_fock: int
    params: np.ndarray
    thetas_per_layer: List[np.ndarray]
    alphas_per_layer: List[complex]
    n_iterations: int
    optimization_time: float


def optimize_snap_d_params(
    target_coeffs: np.ndarray,
    n_fock: int,
    depth: int,
    *,
    n_snap: int = 0,
    n_restarts: int = 2,
    maxiter: int = 300,
    verbose: bool = True,
    seed: int = 1234,
) -> SNAPDResult:
    target = np.zeros(n_fock, dtype=complex)
    target[: len(target_coeffs)] = target_coeffs
    norm = np.linalg.norm(target)
    if norm < 1e-15:
        raise ValueError("Target state has zero norm.")
    target /= norm

    if n_snap <= 0:
        n_snap = len(target_coeffs)
    n_snap = min(n_snap, n_fock)

    params_per_layer = n_snap + 2
    n_params = depth * params_per_layer
    rng = np.random.default_rng(seed)

    best_result = None
    best_fidelity = -1.0
    total_iters = 0
    t_start = time.time()

    for restart in range(n_restarts):
        x0 = np.zeros(n_params)
        for layer in range(depth):
            offset = layer * params_per_layer
            x0[offset : offset + n_snap] = rng.uniform(-np.pi, np.pi, n_snap)
            x0[offset + n_snap] = rng.uniform(-0.5, 0.5)
            x0[offset + n_snap + 1] = rng.uniform(-0.5, 0.5)

        result = minimize(
            fidelity_cost,
            x0,
            args=(target, n_fock, depth, n_snap),
            method="L-BFGS-B",
            options={"maxiter": maxiter, "ftol": 1e-15, "gtol": 1e-10},
        )

        fid = 1.0 - result.fun
        total_iters += result.nit
        if verbose:
            print(f"  SNAP+D restart {restart + 1}/{n_restarts}: fidelity = {fid:.8f}, iters = {result.nit}")

        if fid > best_fidelity:
            best_fidelity = fid
            best_result = result

    params = best_result.x
    thetas_per_layer = []
    alphas_per_layer = []
    for layer in range(depth):
        offset = layer * params_per_layer
        thetas = params[offset : offset + n_snap].copy()
        re_alpha = params[offset + n_snap]
        im_alpha = params[offset + n_snap + 1]
        thetas_per_layer.append(thetas)
        alphas_per_layer.append(complex(re_alpha, im_alpha))

    return SNAPDResult(
        fidelity=best_fidelity,
        depth=depth,
        n_fock=n_fock,
        params=params,
        thetas_per_layer=thetas_per_layer,
        alphas_per_layer=alphas_per_layer,
        n_iterations=total_iters,
        optimization_time=time.time() - t_start,
    )


def apply_snap_d_circuit(qc: CVCircuit, mode, snap_d_result: SNAPDResult) -> None:
    for layer in range(snap_d_result.depth):
        thetas = snap_d_result.thetas_per_layer[layer]
        alpha = snap_d_result.alphas_per_layer[layer]
        for n_idx, theta_val in enumerate(thetas):
            if abs(theta_val) > 1e-12:
                qc.cv_snap(float(theta_val), n_idx, mode)
        qc.cv_d(alpha, mode)


@dataclass
class GivensResult:
    fidelity: float
    n_active: int
    n_fock: int
    rotations: List[Dict[str, float]]
    prepared_state: np.ndarray


def givens_decomposition(target_coeffs: np.ndarray, n_fock: int, verbose: bool = False) -> GivensResult:
    target = np.zeros(n_fock, dtype=complex)
    n_copy = min(len(target_coeffs), n_fock)
    target[:n_copy] = target_coeffs[:n_copy]
    norm = np.linalg.norm(target)
    if norm < 1e-15:
        raise ValueError("Target state has zero norm.")
    target /= norm

    n_active = 1
    for k in range(n_fock - 1, -1, -1):
        if abs(target[k]) > 1e-15:
            n_active = k + 1
            break

    state = target.copy()
    inverse_rotations = []
    for k in range(n_active - 1, 0, -1):
        a = state[k - 1]
        b = state[k]
        r = np.sqrt(abs(a) ** 2 + abs(b) ** 2)

        if r < 1e-15:
            inverse_rotations.append({"n": k - 1, "theta": 0.0, "phi": 0.0})
            continue

        theta = np.arctan2(abs(b), abs(a))
        phi = np.angle(a) - np.angle(b) + np.pi

        c = np.cos(theta)
        s = np.sin(theta)
        new_km1 = c * state[k - 1] - np.exp(1j * phi) * s * state[k]
        new_k = np.exp(-1j * phi) * s * state[k - 1] + c * state[k]
        state[k - 1] = new_km1
        state[k] = new_k

        inverse_rotations.append({"n": k - 1, "theta": float(theta), "phi": float(phi)})

    global_phase = np.angle(state[0])
    prep_rotations = list(reversed(inverse_rotations))
    if abs(global_phase) > 1e-12 and prep_rotations:
        prep_rotations[0] = dict(prep_rotations[0])
        prep_rotations[0]["phi"] = prep_rotations[0]["phi"] + global_phase

    prepared = np.zeros(n_fock, dtype=complex)
    prepared[0] = 1.0
    for rot in prep_rotations:
        n_level = rot["n"]
        theta = rot["theta"]
        phi = rot["phi"]
        c = np.cos(theta)
        s = np.sin(theta)
        a = prepared[n_level]
        b = prepared[n_level + 1]
        prepared[n_level] = c * a + np.exp(1j * phi) * s * b
        prepared[n_level + 1] = -np.exp(-1j * phi) * s * a + c * b

    fidelity = float(np.abs(np.vdot(target, prepared)) ** 2)
    if verbose:
        print(f"  Givens decomposition: n_active = {n_active}, fidelity = {fidelity:.10f}")

    return GivensResult(
        fidelity=fidelity,
        n_active=n_active,
        n_fock=n_fock,
        rotations=prep_rotations,
        prepared_state=prepared,
    )


def givens_resource_stats(givens_result: GivensResult) -> Dict[str, float]:
    n_nontrivial = sum(1 for rot in givens_result.rotations if abs(rot["theta"]) > 1e-12)
    return {
        "n_active_fock_levels": float(givens_result.n_active),
        "n_givens_rotations_total": float(len(givens_result.rotations)),
        "n_givens_rotations_nontrivial": float(n_nontrivial),
        "n_jc_pulses": float(n_nontrivial),
        "analytic_preparation_fidelity": float(givens_result.fidelity),
    }


## Trotterized Hybrid Evolution Circuit

The next cell copies out the gate logic that matters for the paper-level implementation story.

For each Pauli factor $P_j$ in the heat generator:

1. rotate $X$ and $Y$ factors into the $Z$ basis,
2. collect the parity of all active qubits into one target qubit,
3. apply one conditional oscillator displacement,
4. undo the parity ladder and basis rotation.

With the $\hbar=1$ convention,
$$
e^{-i\lambda\hat x} = D\!\left(-\frac{i\lambda}{\sqrt 2}\right),
$$
so a Pauli coefficient $c_j$ over one time slice $\Delta t$ becomes the bosonic displacement amplitude
$$
\alpha_j = -\frac{i c_j \Delta t}{\sqrt 2}.
$$

The `logical_trotter_counts` helper below is useful because it reproduces the per-step counts that should be quoted in the paper narrative, rather than only the full `count_ops()` numbers after state preparation and squeezing have been added.


In [ ]:
# -----------------------------------------------------------------------------
# 3) Gate-level bosonic circuit helpers
# -----------------------------------------------------------------------------

def pad_coefficients(coeffs: np.ndarray, max_fock_level: int) -> np.ndarray:
    out = np.zeros(max_fock_level, dtype=complex)
    out[: len(coeffs)] = coeffs
    return out


def prepare_dv_basis_state(qc: CVCircuit, qreg: QuantumRegister, basis_index: int) -> None:
    n_qubits = len(qreg)
    for qidx in range(n_qubits):
        bit = (basis_index >> (n_qubits - 1 - qidx)) & 1
        if bit == 1:
            qc.x(qreg[qidx])


def _apply_basis_change_forward(qc: CVCircuit, qreg: QuantumRegister, pauli: str) -> None:
    for qidx, ch in enumerate(pauli):
        if ch == "X":
            qc.h(qreg[qidx])
        elif ch == "Y":
            qc.rz(np.pi / 2.0, qreg[qidx])
            qc.h(qreg[qidx])


def _apply_basis_change_inverse(qc: CVCircuit, qreg: QuantumRegister, pauli: str) -> None:
    for qidx in reversed(range(len(pauli))):
        ch = pauli[qidx]
        if ch == "X":
            qc.h(qreg[qidx])
        elif ch == "Y":
            qc.h(qreg[qidx])
            qc.rz(-np.pi / 2.0, qreg[qidx])


def _apply_parity_conditional_displacement(
    qc: CVCircuit,
    mode,
    qreg: QuantumRegister,
    active_z_qubits: Sequence[int],
    alpha: complex,
) -> None:
    if len(active_z_qubits) == 0:
        qc.cv_d(alpha, mode)
        return

    if len(active_z_qubits) == 1:
        qc.cv_c_d(alpha, mode, qreg[active_z_qubits[0]])
        return

    target = active_z_qubits[-1]
    for cidx in active_z_qubits[:-1]:
        qc.cx(qreg[cidx], qreg[target])
    qc.cv_c_d(alpha, mode, qreg[target])
    for cidx in reversed(active_z_qubits[:-1]):
        qc.cx(qreg[cidx], qreg[target])


def apply_pauli_term_trotter_step(
    qc: CVCircuit,
    mode,
    qreg: QuantumRegister,
    pauli: str,
    coeff: complex,
    dt: float,
) -> None:
    lam = coeff * dt
    alpha = -1.0j * lam / np.sqrt(2.0)

    _apply_basis_change_forward(qc, qreg, pauli)
    active = [idx for idx, ch in enumerate(pauli) if ch != "I"]
    _apply_parity_conditional_displacement(qc, mode, qreg, active, alpha)
    _apply_basis_change_inverse(qc, qreg, pauli)


def apply_trotterized_evolution(
    qc: CVCircuit,
    mode,
    qreg: QuantumRegister,
    pauli_strings: Sequence[str],
    coeffs: Sequence[complex],
    total_time: float,
    n_trotter_steps: int,
) -> None:
    dt = total_time / n_trotter_steps
    for _ in range(n_trotter_steps):
        for pauli, coeff in zip(pauli_strings, coeffs):
            apply_pauli_term_trotter_step(qc, mode, qreg, pauli, coeff, dt)


def logical_trotter_counts(pauli_strings: Sequence[str]) -> Dict[str, int]:
    counts = {"cv_d": 0, "cv_c_d": 0, "cx": 0, "h": 0, "rz": 0}
    for pauli in pauli_strings:
        counts["h"] += 2 * sum(ch in ("X", "Y") for ch in pauli)
        counts["rz"] += 2 * sum(ch == "Y" for ch in pauli)

        active = [ch for ch in pauli if ch != "I"]
        if len(active) == 0:
            counts["cv_d"] += 1
        else:
            counts["cv_c_d"] += 1
            if len(active) >= 2:
                counts["cx"] += 2 * (len(active) - 1)
    return counts


def detect_statevector_layout(max_fock_level: int, n_dv_qubits: int) -> str:
    qmr = QumodeRegister(num_qumodes=1, num_qubits_per_qumode=int(np.log2(max_fock_level)))
    qbr = QuantumRegister(n_dv_qubits, "q")
    qc = CVCircuit(qbr, qmr)

    probe = np.zeros(max_fock_level, dtype=complex)
    probe[1] = 1.0
    qc.cv_initialize(probe, qmr[0])

    state, _, _ = cv_util.simulate(qc, shots=1, return_fockcounts=False, add_save_statevector=True)
    vec = np.asarray(state.data, dtype=complex)
    idx = int(np.where(np.abs(vec) > 1e-12)[0][0])
    return "fock_major" if idx == 2 ** n_dv_qubits else "qubit_major"


def qiskit_to_physics_index_map(n_qubits: int) -> np.ndarray:
    return np.array([int(f"{i:0{n_qubits}b}"[::-1], 2) for i in range(2 ** n_qubits)], dtype=int)


def extract_postselected_dv(
    statevector: np.ndarray,
    *,
    layout: str,
    max_fock_level: int,
    n_dv_qubits: int,
) -> np.ndarray:
    dv_dim = 2 ** n_dv_qubits
    if layout == "fock_major":
        dv_qiskit = statevector[:dv_dim]
    else:
        dv_qiskit = statevector[0::max_fock_level][:dv_dim]
    return dv_qiskit[qiskit_to_physics_index_map(n_dv_qubits)]


def circuit_resource_report(qc: CVCircuit) -> Dict[str, object]:
    return {
        "depth": int(qc.depth()),
        "size": int(qc.size()),
        "count_ops": dict(qc.count_ops()),
    }


def build_heat_lchs_circuit(
    pauli_strings: Sequence[str],
    coeffs: Sequence[complex],
    coeffs_seed: np.ndarray,
    *,
    total_time: float,
    n_trotter_steps: int,
    max_fock_level: int,
    r_target: float,
    r_prime: float,
    init_basis_index: int,
    state_prep_method: str = "injection",
    snap_depth: int = 4,
    snap_restarts: int = 2,
    snap_maxiter: int = 300,
    random_seed: int = 1234,
):
    if max_fock_level & (max_fock_level - 1):
        raise ValueError("max_fock_level must be a power of 2 for bosonic_qiskit.")
    if len(coeffs_seed) > max_fock_level:
        raise ValueError("max_fock_level must be at least the number of active coefficients.")

    n_qubits = len(pauli_strings[0])
    qmr = QumodeRegister(num_qumodes=1, num_qubits_per_qumode=int(math.log2(max_fock_level)))
    qbr = QuantumRegister(n_qubits, "q")
    qc = CVCircuit(qbr, qmr)
    mode = qmr[0]

    if state_prep_method == "injection":
        qc.cv_initialize(pad_coefficients(coeffs_seed, max_fock_level), mode)
        prep_meta = {
            "stateprep_method": "injection",
            "stateprep_seed_fidelity": 1.0,
        }
    elif state_prep_method == "snap_d":
        snap_result = optimize_snap_d_params(
            coeffs_seed,
            n_fock=max_fock_level,
            depth=snap_depth,
            n_restarts=snap_restarts,
            maxiter=snap_maxiter,
            verbose=True,
            seed=random_seed,
        )
        apply_snap_d_circuit(qc, mode, snap_result)
        prep_meta = {
            "stateprep_method": "snap_d",
            "stateprep_seed_fidelity": float(snap_result.fidelity),
            "stateprep_depth": float(snap_depth),
            "stateprep_restarts": float(snap_restarts),
            "stateprep_optimization_time_s": float(snap_result.optimization_time),
        }
    elif state_prep_method == "givens":
        givens_result = givens_decomposition(coeffs_seed, n_fock=max_fock_level, verbose=True)
        qc.cv_initialize(pad_coefficients(givens_result.prepared_state, max_fock_level), mode)
        prep_meta = {
            "stateprep_method": "givens_injected",
            **givens_resource_stats(givens_result),
            "note": "Analytic Law-Eberly angles are computed exactly, but the simulator realization uses state injection because selective JC pulses are not exposed by bosonic_qiskit.",
        }
    else:
        raise ValueError("state_prep_method must be one of: injection, snap_d, givens")

    if abs(r_prime) > 1e-14:
        qc.cv_sq(-r_prime, mode)

    prepare_dv_basis_state(qc, qbr, init_basis_index)
    apply_trotterized_evolution(qc, mode, qbr, pauli_strings, coeffs, total_time, n_trotter_steps)

    if abs(r_target) > 1e-14:
        qc.cv_sq(+r_target, mode)

    return qc, prep_meta


def run_heat_lchs_method(
    pauli_strings: Sequence[str],
    coeffs: Sequence[complex],
    coeffs_seed: np.ndarray,
    u_ref: np.ndarray,
    *,
    total_time: float,
    n_trotter_steps: int,
    max_fock_level: int,
    r_target: float,
    r_prime: float,
    init_basis_index: int,
    state_prep_method: str,
    snap_depth: int = 4,
    snap_restarts: int = 2,
    snap_maxiter: int = 300,
    random_seed: int = 1234,
) -> Dict[str, object]:
    qc, prep_meta = build_heat_lchs_circuit(
        pauli_strings,
        coeffs,
        coeffs_seed,
        total_time=total_time,
        n_trotter_steps=n_trotter_steps,
        max_fock_level=max_fock_level,
        r_target=r_target,
        r_prime=r_prime,
        init_basis_index=init_basis_index,
        state_prep_method=state_prep_method,
        snap_depth=snap_depth,
        snap_restarts=snap_restarts,
        snap_maxiter=snap_maxiter,
        random_seed=random_seed,
    )

    state, _, _ = cv_util.simulate(qc, shots=1, return_fockcounts=False, add_save_statevector=True)
    statevector = np.asarray(state.data, dtype=complex)

    layout = detect_statevector_layout(max_fock_level, len(pauli_strings[0]))
    dv_vec = extract_postselected_dv(
        statevector,
        layout=layout,
        max_fock_level=max_fock_level,
        n_dv_qubits=len(pauli_strings[0]),
    )

    eta, rel_err = fit_global_scale(dv_vec, u_ref)
    return {
        "dv_bosonic": dv_vec,
        "layout": layout,
        "post_prob": float(np.real(np.vdot(dv_vec, dv_vec))),
        "vs_classical_rel_error": float(rel_err),
        "vs_classical_fidelity": float(state_fidelity(dv_vec, u_ref)),
        "eta_vs_classical": eta,
        "resources": circuit_resource_report(qc),
        "prep_meta": prep_meta,
    }


def print_result_summary(name: str, result: Dict[str, object]) -> None:
    print(f"\n=== {name} ===")
    print(f"layout                     : {result['layout']}")
    print(f"post-selection probability : {result['post_prob']:.6e}")
    print(f"relative error vs classical: {result['vs_classical_rel_error']:.6e}")
    print(f"fidelity vs classical      : {result['vs_classical_fidelity']:.8f}")
    print(f"best-fit scale eta         : {result['eta_vs_classical']}")
    print(f"prep metadata              : {result['prep_meta']}")
    print(f"depth / size               : {result['resources']['depth']} / {result['resources']['size']}")
    print(f"count_ops                  : {result['resources']['count_ops']}")


## Experiment Configuration

The next cell exposes all numerical settings in one place.

The defaults below are chosen to be a balanced notebook configuration rather than the heaviest paper-style setting. In practice:

- increase `n_trotter_steps` to reduce Trotter error,
- increase `n_coeff` and `max_fock_level` to improve the CV seed representation,
- increase `snap_depth`, `snap_restarts`, and `snap_maxiter` to improve SNAP+D seed-preparation fidelity.

If you want a closer paper-style run, the active repo has often used `max_fock_level = 64`, `n_coeff = 24` or `48`, and `n_trotter_steps = 100` or `200`.


In [ ]:
# Heat-equation parameters
alpha = 1.0
h_grid = 1.0
total_time = 1.0
init_basis_index = 1  # |01>

# LCHS kernel parameters (hbar = 1 convention)
r_target = 5.0
r_prime = 4.0
beta = 0.30
n_coeff = 24
n_quad = 220

# Circuit parameters
max_fock_level = 64
n_trotter_steps = 40
snap_depth = 4
snap_restarts = 2
snap_maxiter = 300
random_seed = 1234

methods = ("injection", "snap_d", "givens")

pauli_strings, coeffs, A = heat_system_dirichlet_2qubit(alpha=alpha, h_grid=h_grid)
u0 = np.zeros(A.shape[0], dtype=complex)
u0[init_basis_index] = 1.0
u_ref = classical_heat_solution(A, total_time, u0)
coeffs_seed = lchs_coefficients_explicit_overlap(
    r_target=r_target,
    r_prime=r_prime,
    beta=beta,
    n_coeff=n_coeff,
    n_quad=n_quad,
)

print("Discrete Dirichlet heat generator A:")
print(np.real_if_close(A))

print("\nPer-step logical Trotter counts:")
print(logical_trotter_counts(pauli_strings))

print("\nFirst eight seed-coefficient magnitudes:")
print(np.abs(coeffs_seed[:8]))
print("Seed-state norm:", np.linalg.norm(coeffs_seed))

print("\nExact classical reference vector e^{-AT} u(0):")
print(np.real_if_close(u_ref))


In [ ]:
results = {}
for method in methods:
    result = run_heat_lchs_method(
        pauli_strings,
        coeffs,
        coeffs_seed,
        u_ref,
        total_time=total_time,
        n_trotter_steps=n_trotter_steps,
        max_fock_level=max_fock_level,
        r_target=r_target,
        r_prime=r_prime,
        init_basis_index=init_basis_index,
        state_prep_method=method,
        snap_depth=snap_depth,
        snap_restarts=snap_restarts,
        snap_maxiter=snap_maxiter,
        random_seed=random_seed,
    )
    results[method] = result
    print_result_summary(method, result)

print("\nPairwise shape fidelities between bosonic outputs:")
for i, method_i in enumerate(methods):
    for method_j in methods[i + 1 :]:
        fid = state_fidelity(results[method_i]["dv_bosonic"], results[method_j]["dv_bosonic"])
        print(f"  {method_i:9s} vs {method_j:9s}: {fid:.8f}")


## How To Read The Output

- `injection` is the clean baseline: it isolates the Trotterization and post-selection effects without state-preparation error.
- `snap_d` adds a genuine gate-level CV state-preparation stage. If its fidelity is already near 1, then the remaining discrepancy is dominated by Trotter error and finite Fock truncation.
- `givens` reports the Law-Eberly resource counts that would matter on hardware with number-selective JC control, but its statevector here is still injected because that selective gate is not presently available in `bosonic_qiskit`.

This notebook therefore matches the working-paper story if you phrase it precisely:

1. the active numerical experiment is the 2-qubit homogeneous Dirichlet heat equation,
2. the hybrid evolution is compiled by Pauli-term Trotterization using parity-encoded conditional displacements,
3. SNAP+D is the fully gate-level seed-state path,
4. Law-Eberly is currently included as an analytic resource model plus an exact simulator-state injection.
